In [17]:
import pandas as pd
import numpy as np

# naya combined aqi + weather data
df = pd.read_csv("../data/raw/aqi_hourly_v2.csv", index_col=0, parse_dates=True)

# chhote gaps (6 ghante tak) bhar diye, lambe waise hi chhode
df["AQI"] = df["AQI"].interpolate(limit=6, limit_area="inside")
print(df.shape, df["AQI"].isna().mean().round(3))

(74544, 7) 0.159


In [18]:
# 24 ghante se lambe gaps kahan hain
aqi = df["AQI"]
is_na = aqi.isna()
blocks = (is_na != is_na.shift()).cumsum()
gap_len = is_na.groupby(blocks).sum()

for b in gap_len[gap_len > 24].index:
    idx = aqi[blocks == b].index
    print(idx.min(), "to", idx.max(), "->", len(idx), "hours")

2017-07-01 00:00:00 to 2017-08-23 10:00:00 -> 1283 hours
2017-08-25 21:00:00 to 2017-08-30 15:00:00 -> 115 hours
2017-09-01 01:00:00 to 2017-09-30 23:00:00 -> 719 hours
2017-11-18 06:00:00 to 2017-12-01 10:00:00 -> 317 hours
2017-12-31 02:00:00 to 2018-01-01 11:00:00 -> 34 hours
2018-03-15 04:00:00 to 2018-03-16 15:00:00 -> 36 hours
2018-03-18 21:00:00 to 2018-03-20 23:00:00 -> 51 hours
2018-04-15 09:00:00 to 2018-04-19 13:00:00 -> 101 hours
2018-04-22 09:00:00 to 2018-05-01 20:00:00 -> 228 hours
2018-05-17 08:00:00 to 2018-05-21 18:00:00 -> 107 hours
2018-05-24 22:00:00 to 2018-05-30 11:00:00 -> 134 hours
2018-06-02 10:00:00 to 2018-06-11 03:00:00 -> 210 hours
2018-06-11 21:00:00 to 2018-06-13 18:00:00 -> 46 hours
2018-06-16 20:00:00 to 2018-06-18 13:00:00 -> 42 hours
2018-06-23 19:00:00 to 2018-06-25 00:00:00 -> 30 hours
2018-07-02 00:00:00 to 2018-07-03 10:00:00 -> 35 hours
2018-07-17 23:00:00 to 2018-07-20 10:00:00 -> 60 hours
2018-08-19 00:00:00 to 2018-08-20 14:00:00 -> 39 hours


In [19]:
feat = pd.DataFrame(index=df.index)

# pichhle ghanto ka aqi
feat["aqi_now"] = aqi
for l in [1, 2, 3, 6, 12, 24, 48, 72]:
    feat[f"aqi_lag_{l}"] = aqi.shift(l)

# rolling average (thoda missing chal jayega)
feat["aqi_roll_24"] = aqi.rolling(24, min_periods=18).mean()
feat["aqi_roll_72"] = aqi.rolling(72, min_periods=54).mean()

# weather
for c in ["temperature_2m", "relative_humidity_2m", "wind_speed_10m",
          "wind_direction_10m", "surface_pressure", "precipitation"]:
    feat[c] = df[c]

# time ke features
feat["hour"] = df.index.hour
feat["month"] = df.index.month
feat["dayofweek"] = df.index.dayofweek

In [20]:
horizon = 24
data = feat.copy()
data["target"] = aqi.shift(-horizon)
data = data.dropna()
print(data.shape)

# time ke hisaab se split: 2025 poora test
train = data[data.index < "2025-01-01"]
test = data[data.index >= "2025-01-01"]
print(train.shape, test.shape)

X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]

(50618, 21)
(43676, 21) (6942, 21)


In [21]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train)

# aqi 0-500 ke beech hi hota hai
pred = np.clip(model.predict(X_test), 0, 500)

# persistence: kal isi time ka aqi aaj jaisa hi rahega
pers = X_test["aqi_now"]

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

print("xgboost     mae:", round(mean_absolute_error(y_test, pred), 2), " rmse:", round(rmse(y_test, pred), 2))
print("persistence mae:", round(mean_absolute_error(y_test, pers), 2), " rmse:", round(rmse(y_test, pers), 2))

xgboost     mae: 45.85  rmse: 63.21
persistence mae: 51.96  rmse: 76.27


In [22]:
# test ke har mahine mein error kitna hai
res = pd.DataFrame({"y": y_test, "xgb": pred, "pers": pers})
res["xgb_err"] = (res["y"] - res["xgb"]).abs()
res["pers_err"] = (res["y"] - res["pers"]).abs()
print(res.groupby(res.index.month)[["y", "xgb_err", "pers_err"]].mean().round(1))

               y  xgb_err  pers_err
Timestamp                          
1          257.1     66.6      75.4
2          163.1     46.9      52.0
3          140.5     39.7      42.5
4          168.3     43.6      51.2
5          135.9     46.8      55.7
6          101.7     29.4      30.4
7           61.4     22.8      27.7
9           81.7     46.9      57.7
10         202.3     50.1      56.5
11         326.4     47.2      53.8
12         313.8     56.3      64.5


In [23]:
weather_cols = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m",
                "wind_direction_10m", "surface_pressure", "precipitation"]

def run(horizon, future_weather=False, test_year=2025):
    d = feat.copy()
    if future_weather:
        # horizon ghante baad ka weather bhi feature bana diya
        for c in weather_cols:
            d[c + "_fut"] = df[c].shift(-horizon)
    d["target"] = aqi.shift(-horizon)
    d = d.dropna()

    # test year se pehle ke saal train, wo saal test
    tr = d[d.index.year < test_year]
    te = d[d.index.year == test_year]
    Xtr, ytr = tr.drop(columns="target"), tr["target"]
    Xte, yte = te.drop(columns="target"), te["target"]

    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
    m.fit(Xtr, ytr)
    p = np.clip(m.predict(Xte), 0, 500)
    pers = Xte["aqi_now"]

    return {"horizon": horizon,
            "xgb_mae": mean_absolute_error(yte, p), "pers_mae": mean_absolute_error(yte, pers),
            "xgb_rmse": rmse(yte, p), "pers_rmse": rmse(yte, pers)}

rows = [run(h) for h in [6, 12, 24, 48, 72]]
print(pd.DataFrame(rows).round(1))

   horizon  xgb_mae  pers_mae  xgb_rmse  pers_rmse
0        6     40.6      53.3      55.7       74.7
1       12     43.6      61.7      60.5       85.2
2       24     45.9      52.0      63.2       76.3
3       48     50.5      62.2      68.2       87.2
4       72     54.6      66.9      72.8       92.9


In [24]:
rows = [run(h, future_weather=True) for h in [6, 12, 24, 48, 72]]
print(pd.DataFrame(rows).round(1))

   horizon  xgb_mae  pers_mae  xgb_rmse  pers_rmse
0        6     40.2      53.3      55.2       74.7
1       12     43.0      61.7      59.7       85.2
2       24     44.5      52.0      61.5       76.3
3       48     47.1      62.2      64.1       87.2
4       72     49.6      66.9      67.2       92.9


In [25]:
res = pd.DataFrame([run(24, test_year=2023), run(24, test_year=2025)], index=[2023, 2025])
print(res.round(1))

      horizon  xgb_mae  pers_mae  xgb_rmse  pers_rmse
2023       24     50.9      55.5      68.5       81.5
2025       24     45.9      52.0      63.2       76.3


In [26]:
def run_delta(horizon, test_year=2025):
    d = feat.copy()
    # target ab aqi ka change hai, aage ka aqi nahi
    d["target"] = aqi.shift(-horizon) - aqi
    d = d.dropna()

    tr = d[d.index.year < test_year]
    te = d[d.index.year == test_year]
    Xtr, ytr = tr.drop(columns="target"), tr["target"]
    Xte, yte = te.drop(columns="target"), te["target"]

    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
    m.fit(Xtr, ytr)

    # change predict kiya, abhi ke aqi mein jod ke asli aqi bana liya
    p = np.clip(Xte["aqi_now"] + m.predict(Xte), 0, 500)
    y_true = Xte["aqi_now"] + yte
    pers = Xte["aqi_now"]

    return {"horizon": horizon,
            "xgb_mae": mean_absolute_error(y_true, p), "pers_mae": mean_absolute_error(y_true, pers),
            "xgb_rmse": rmse(y_true, p), "pers_rmse": rmse(y_true, pers)}

rows = [run_delta(h) for h in [6, 12, 24, 48, 72]]
print(pd.DataFrame(rows).round(1))

   horizon  xgb_mae  pers_mae  xgb_rmse  pers_rmse
0        6     40.3      53.3      55.5       74.7
1       12     44.1      61.7      61.2       85.2
2       24     45.9      52.0      63.1       76.3
3       48     50.6      62.2      68.1       87.2
4       72     54.7      66.9      72.9       92.9


In [27]:
# ek ghante mein aur 24 ghante mein aqi kitna badalta hai (saal ke hisaab se)
c1 = aqi.diff().abs()
c24 = aqi.diff(24).abs()
out = pd.DataFrame({"1h_change": c1.groupby(c1.index.year).mean(),
                    "24h_change": c24.groupby(c24.index.year).mean()})
print(out.round(2))

           1h_change  24h_change
Timestamp                       
2017           26.07       66.68
2018           22.52       57.44
2019           21.52       63.24
2020           21.93       54.73
2021           19.23       52.51
2022           20.97       54.22
2023           19.65       52.66
2024           21.12       52.34
2025           22.18       51.71


In [28]:
import os, json
os.makedirs("models", exist_ok=True)

# grap daily average pe chalta hai, to agle 1, 2, 3 din ka average aqi predict karenge
daily = aqi.rolling(24, min_periods=18).mean()
targets = {"day1": daily.shift(-24), "day2": daily.shift(-48), "day3": daily.shift(-72)}

def stage_idx(a):
    # 0 = koi stage nahi, 1 se 4 = stage i se iv (pdf ke thresholds)
    return np.digitize(a, [200, 300, 400, 450], right=True)

rows = []
for name, tgt in targets.items():
    d = feat.copy()
    d["target"] = tgt
    d = d.dropna()

    # 2025 se pehle train, 2025 test
    tr = d[d.index.year < 2025]
    te = d[d.index.year == 2025]
    Xtr, ytr = tr.drop(columns="target"), tr["target"]
    Xte, yte = te.drop(columns="target"), te["target"]

    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
    m.fit(Xtr, ytr)
    p = np.clip(m.predict(Xte), 0, 500)

    # baseline: pichhle 24 ghante ka average hi aage bhi rahega
    base = Xte["aqi_roll_24"]

    rows.append({"day": name,
                 "mae": mean_absolute_error(yte, p),
                 "base_mae": mean_absolute_error(yte, base),
                 "stage_acc": (stage_idx(p) == stage_idx(yte.values)).mean(),
                 "base_stage_acc": (stage_idx(base.values) == stage_idx(yte.values)).mean()})

    m.save_model(f"models/xgb_{name}.json")

# feature columns ka order bhi save, dashboard mein wahi chahiye
json.dump(list(Xtr.columns), open("models/feature_cols.json", "w"))
print(pd.DataFrame(rows).round(3))

    day     mae  base_mae  stage_acc  base_stage_acc
0  day1  27.776    32.331      0.822           0.792
1  day2  37.463    44.469      0.756           0.716
2  day3  40.753    50.898      0.747           0.677


In [30]:
from sklearn.metrics import confusion_matrix

names = ["none", "I", "II", "III", "IV"]

for name, tgt in targets.items():
    d = feat.copy()
    d["target"] = tgt
    d = d.dropna()
    te = d[d.index.year == 2025]
    Xte, yte = te.drop(columns="target"), te["target"]

    # saved model load karke predict
    m = XGBRegressor()
    m.load_model(f"../models/xgb_{name}.json")
    p = np.clip(m.predict(Xte), 0, 500)

    true_s, pred_s = stage_idx(yte.values), stage_idx(p)

    # rows = asli stage, columns = predicted stage
    cm = confusion_matrix(true_s, pred_s, labels=range(5))
    print(name)
    print(pd.DataFrame(cm, index=names, columns=names))

    # sirf sardi ke mahine (oct se feb)
    w = te.index.month.isin([10, 11, 12, 1, 2])
    print("winter stage acc:", round((true_s[w] == pred_s[w]).mean(), 3), "| rows:", w.sum())
    print("-----")

day1
      none    I   II  III  IV
none  3835  368   18    0   0
I      202  925  290    0   0
II      15  248  928   14   0
III      0    1   69   20   0
IV       0    0    8    7   0
winter stage acc: 0.719 | rows: 3600
-----
day2
      none    I   II  III  IV
none  3841  275   54    0   0
I      353  706  329    6   0
II      20  533  651    0   0
III      0   21   69    0   0
IV       0    9    6    0   0
winter stage acc: 0.608 | rows: 3576
-----
day3
      none    I   II  III  IV
none  3814  277   31    0   0
I      388  682  318    3   0
II      44  553  586    0   0
III      0   40   50    0   0
IV       0   15    0    0   0
winter stage acc: 0.589 | rows: 3552
-----


In [33]:
rows = []
for name, tgt in targets.items():
    d = feat.copy()
    d["target"] = tgt
    d = d.dropna()
    tr = d[d.index.year < 2025]
    te = d[d.index.year == 2025]
    Xtr, ytr = tr.drop(columns="target"), tr["target"]
    Xte, yte = te.drop(columns="target"), te["target"]

    # pehle wala median model (saved)
    m50 = XGBRegressor()
    m50.load_model(f"../models/xgb_{name}.json")
    p50 = np.clip(m50.predict(Xte), 0, 500)

    # upar ke 80th percentile wala naya model
    m80 = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                       subsample=0.8, colsample_bytree=0.8, random_state=42,
                       objective="reg:quantileerror", quantile_alpha=0.8)
    m80.fit(Xtr, ytr)
    p80 = np.clip(m80.predict(Xte), 0, 500)
    m80.save_model(f"../models/xgb_{name}_q80.json")

    base = Xte["aqi_roll_24"].values
    true_s = stage_idx(yte.values)
    w = te.index.month.isin([10, 11, 12, 1, 2])

    for label, p in [("median", p50), ("q80", p80), ("base", base)]:
        ps = stage_idx(p)
        t_w, p_w = true_s >= 2, ps >= 2      # warning = stage ii ya upar
        t_s, p_s = true_s >= 3, ps >= 3      # severe = stage iii ya upar
        rows.append({"day": name, "model": label,
                     "winter_acc": (true_s[w] == ps[w]).mean(),
                     "within_1": (np.abs(true_s - ps) <= 1).mean(),
                     "warn_recall": (t_w & p_w).sum() / t_w.sum(),
                     "warn_prec": (t_w & p_w).sum() / max(p_w.sum(), 1),
                     "severe_recall": (t_s & p_s).sum() / t_s.sum(),
                     "mae": mean_absolute_error(yte, p)})

print(pd.DataFrame(rows).round(3))

    day   model  winter_acc  within_1  warn_recall  warn_prec  severe_recall  \
0  day1  median       0.719     0.994        0.798      0.773          0.257   
1  day1     q80       0.569     0.986        0.913      0.596          0.486   
2  day1    base       0.666     0.985        0.708      0.713          0.495   
3  day2  median       0.608     0.983        0.555      0.651          0.000   
4  day2     q80       0.464     0.976        0.869      0.536          0.067   
5  day2    base       0.562     0.961        0.588      0.603          0.067   
6  day3  median       0.589     0.980        0.494      0.644          0.000   
7  day3     q80       0.488     0.969        0.852      0.536          0.000   
8  day3    base       0.510     0.952        0.530      0.544          0.000   

      mae  
0  27.776  
1  42.033  
2  32.331  
3  37.463  
4  51.916  
5  44.469  
6  40.753  
7  54.735  
8  50.898  
